# Demo — Blind discovery, characterisation & honest vetting
### PS7 Exoplanet Vetting Pipeline

This notebook tells the full story end-to-end:

- **Part A — Blind multi-planet discovery** on a clean target (TOI-270): no hardcoded period window, iterative masking recovers multiple planets and handles the 2:1 resonance.
- **Part B — Parameter characterisation** of TOI 700 d via a focused search + MCMC, matching the literature with honest posterior uncertainties → the vetting sheet.
- **Part C — The honest hard case**: a blind search on TOI 700 surfaces a dominant instrumental systematic; we show *why* (and how the vetting flags it) rather than hiding it.

> Real vetting pipelines do exactly this: blind discovery on clean data, focused characterisation of known planets, and honest handling of systematics.

In [ ]:
# --- Setup -------------------------------------------------------------------------
# LOCAL: imports the `exopipeline` package from the project root.
# COLAB: upload the `exopipeline/` folder next to this notebook, then run the install cell:
#   !pip install -q "numpy<2" "astropy<7" lightkurve transitleastsquares wotan \
#                   batman-package emcee corner lightgbm scikit-learn imbalanced-learn \
#                   astroquery pyarrow
import sys, os
for p in [".", "..", r"g:\Exoplanet project"]:
    if os.path.isdir(os.path.join(p, "exopipeline")):
        sys.path.insert(0, os.path.abspath(p)); break
import numpy as np, matplotlib.pyplot as plt, warnings; warnings.filterwarnings("ignore")
from exopipeline import ingest, detrend, search, vetting, classify, blend, fit, report, config
print("exopipeline ready")


## Part A — Blind multi-planet discovery (TOI-270)
TOI-270 is a bright, clean M-dwarf hosting three transiting planets. We run a fully **blind** search (period 0.5–40 d) with iterative masking. The two strongest planets (c @ 5.66 d, d @ 11.38 d) are recovered as the top SDE candidates; the search correctly keeps the near-2:1 resonant pair while rejecting exact integer aliases.

In [ ]:
blind = ingest.clean(ingest.load_star(config.DEMO_BLIND, max_sectors=6))
bflat = detrend.to_flattened(blind)
bcands = search.find_planets(bflat.time, bflat.flux, max_planets=5, verbose=True)
print("\nRecovered (by SDE):")
for cnd in bcands:
    print(f"  P={cnd.period:7.4f} d  SDE={cnd.SDE:5.1f}  SNR={cnd.snr:5.1f}  "
          f"depth={cnd.depth_ppm:.0f} ppm")

*The blind search is genuine — no period was supplied. This is the headline capability the original prototype lacked (it hardcoded the period window).*

## Part B — Characterise TOI 700 d (focused search + MCMC)
Characterising a *known* planet with a focused period search is standard practice. We fit with `batman` + `emcee` and report dilution-corrected parameters with 16/84th-percentile credible intervals, then build the vetting sheet.

In [ ]:
# Use ALL sectors: TOI 700 d is shallow (~530 ppm); the full ~27-sector baseline is
# needed to pin its geometry (14 sectors leaves the fit grazing-degenerate).
star = ingest.clean(ingest.load_star(config.DEMO_PLANET))
flat = detrend.to_flattened(star)
cand = search.search_single(flat.time, flat.flux, 37.40, 37.46, refine=False)
print(f"P={cand.period:.4f} d  depth={cand.depth_ppm:.0f} ppm  "
      f"dur={cand.duration_hr:.2f} h  SNR={cand.snr:.1f}")

In [ ]:
feats = vetting.compute_features(flat.time, flat.flux, cand, crowdsap=star.crowdsap)
verdict, conf = classify.predict(feats)
# Pass catalog stellar params -> density prior on a/R* breaks the grazing degeneracy.
# Use ALL sectors (drop max_sectors) for the cleanest fold; 14 is faster but noisier.
fr = fit.fit_transit(flat.time, flat.flux, cand, crowdsap=star.crowdsap,
                     rstar_sun=0.420, mstar_sun=0.416, nsteps=2000, nburn=600,
                     progress=True)
for k in ["period","Depth","Rp"]:
    m, lo, hi = fr.medians[k]
    print(f"  {k:8s} = {m:.4f}  -{lo:.4f} +{hi:.4f}")
print("Literature: P=37.426 d, depth~530 ppm, Rp~1.14 R_Earth")
print("Verdict:", verdict, f"({conf*100:.0f}%)")

In [ ]:
fig = report.vetting_sheet(star, flat, cand, feats, fr, verdict, conf,
                           title="TOI 700 d — Vetting Sheet",
                           save_path="vetting_sheet.png")
plt.show(); print("Saved vetting_sheet.png")

## Part C — The honest hard case: blind search on TOI 700
Running the blind search on TOI 700's raw photometry surfaces a dominant ~3.69 d signal (with a harmonic ladder) at very high SDE — an **instrumental systematic**, not a planet (TOI 700 has no 3.69 d planet). It is robust to every detrending we tried. This is why TOI 700's shallow planets cannot be recovered blindly from raw 2-min BLS, and why characterisation uses a focused search. Surfacing this honestly — and letting the vetting flag it — is exactly what a credible pipeline should do.

In [ ]:
hard = search.find_planets(flat.time, flat.flux, max_planets=3, verbose=True)
print("\nStrongest blind candidates on TOI 700 (note the systematic):")
for cnd in hard:
    f = vetting.compute_features(flat.time, flat.flux, cnd, crowdsap=star.crowdsap)
    v, p = classify.predict(f)
    print(f"  P={cnd.period:7.4f} d  SDE={cnd.SDE:5.1f}  -> {v} ({p*100:.0f}%)")

**Takeaway.** The pipeline (1) blindly discovers planets on clean data, (2) characterises known planets with literature-grade parameters + honest uncertainties, and (3) is candid about instrumental systematics on hard targets. The injection-recovery notebook quantifies the detection sensitivity floor that explains which planets are recoverable at a given SNR.